# Chapter 4 assignment: PINNs for initial value problems

**Learning goals.** After this assignment you can

1. reproduce the hand calculation of Section *A PINN computed by hand* with automatic differentiation;
2. implement the soft and the hard treatment of an initial condition and recognize the plateau that the
   trivial equilibrium creates for the logistic equation;
3. solve a second-order IVP (van der Pol) with a trial solution that satisfies both initial conditions
   exactly, and validate it against a classical reference solution.

Everything runs on a CPU in a few minutes. Run the notebook from the repository root (or add the root
to `sys.path`, as in the first code cell). Cells marked `# TODO` must be completed; the *check* cells test
your answers.

## Recap

For $y^{(n)} = f(t, y, \dots, y^{(n-1)})$, $y^{(k)}(t_0)=\eta_k$, a PINN represents the solution by a network
$N_\theta$, either directly (*soft* initial conditions, penalized in the loss) or through the trial solution

$$y_\theta(t) = \sum_{k=0}^{n-1} \eta_k \frac{(t-t_0)^k}{k!} + (t-t_0)^n N_\theta(t) \qquad (\text{hard}),$$

and minimizes the mean squared residual $\mathcal{R}_\theta = y_\theta^{(n)} - f(t, y_\theta, \dots)$ at
collocation points $t_1,\dots,t_M$ (plus the penalty terms in the soft case). Derivatives with respect to $t$
and the gradient with respect to $\theta$ are both computed by automatic differentiation.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "pinnode").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import torch
from pinnode import (MLP, ConstrainedModel, derivative, to_tensor, train, uniform, rel_l2,
                     configure_torch, set_seed)
from pinnode.reference import ivp_reference
configure_torch("float64", threads=1)
set_seed(1234)

## Task 1 - the hand calculation with automatic differentiation

Network: $y_\theta(t) = v_1\tanh(w_1t+b_1) + v_2\tanh(w_2t+b_2) + c$ with $w=(0.1,0.2)$, $b=(0.1,0.2)$,
$v=(0.5,0.6)$, $c=0.1$. Problem $y'=y$, $y(0)=1$, one collocation point $t_1=0.5$, loss
$\mathcal{L} = (y_\theta'(t_1)-y_\theta(t_1))^2 + (y_\theta(0)-1)^2$.

Compute $y_\theta(t_1)$, $y_\theta'(t_1)$ (with `torch.autograd.grad`, `create_graph=True`), the loss, and the
gradient of the loss with respect to all parameters (with `loss.backward()`).

In [ ]:
w = torch.tensor([0.1, 0.2], requires_grad=True)
b = torch.tensor([0.1, 0.2], requires_grad=True)
v = torch.tensor([0.5, 0.6], requires_grad=True)
c = torch.tensor(0.1, requires_grad=True)

def y_theta(t):
    # t: tensor of shape (N, 1); returns shape (N, 1)
    return (v * torch.tanh(w * t + b)).sum(dim=1, keepdim=True) + c

t1 = torch.tensor([[0.5]], requires_grad=True)
t0 = torch.tensor([[0.0]])

y1 = y_theta(t1)
dy1 = ...        # TODO: derivative of y1 with respect to t1 (keep the graph!)
loss = ...       # TODO: residual term + initial-condition term
# TODO: compute the gradient of the loss with loss.backward()
print(float(y1), float(dy1), float(loss))
print("dL/dw =", w.grad, " dL/db =", b.grad, " dL/dv =", v.grad, " dL/dc =", c.grad)

In [ ]:
# check your answer (values of Table ivp:tab:manual)
assert abs(float(y1) - 0.349230) < 1e-6
assert abs(float(dy1) - 0.158708) < 1e-6
assert abs(float(loss) - 0.571743) < 1e-6
assert abs(float(w.grad[0]) + 0.0903757) < 1e-6
assert abs(float(c.grad) + 1.082438) < 1e-6
print("Task 1: correct")

**Question.** Perform one step of `torch.optim.Adam([w, b, v, c], lr=1e-3)` and print the parameters.
Why does every parameter change by (almost exactly) $10^{-3}$? Which parameters are driven only by the
residual term, and why?

In [ ]:
# TODO: one Adam step and a print of the new parameters


## Task 2 - the logistic equation: soft versus hard initial condition

$y' = 2y(1-y)$, $y(0)=0.1$, $t\in[0,2]$, exact solution $y = 0.1e^{2t}/(1+0.1(e^{2t}-1))$.
Complete `make_model` (soft: the plain network; hard: $y_\theta = 0.1 + t\,N_\theta(t)$) and `loss_fn`
(soft: residual + penalty; hard: residual only). Train both formulations with the **sigmoid** activation,
2000 Adam iterations and no L-BFGS, and record the loss at the end.

In [ ]:
R, Y0, T = 2.0, 0.1, 2.0
exact = lambda t: Y0 * np.exp(R * t) / (1 + Y0 * (np.exp(R * t) - 1))
t_col = to_tensor(uniform(100, 0.0, T), requires_grad=True)
t_test = np.linspace(0.0, T, 1001)

def make_model(hard, activation="sigmoid", seed=1234):
    torch.manual_seed(seed)
    net = MLP(1, 1, width=32, depth=3, activation=activation, domain=(0.0, T))
    if hard:
        return ...   # TODO: ConstrainedModel with the transform  y0 + t * N(t)
    return net

def make_loss(model, hard):
    def loss_fn():
        y = model(t_col)
        r = ...      # TODO: residual y' - R y (1 - y)   (use pinnode.derivative)
        L_r = (r ** 2).mean()
        if hard:
            return L_r, {"residual": L_r}
        L_ic = ...   # TODO: (y_theta(0) - Y0)^2
        return L_r + L_ic, {"residual": L_r, "ic": L_ic}
    return loss_fn

results = {}
for hard in (False, True):
    model = make_model(hard)
    hist = train(model.parameters(), make_loss(model, hard), adam_iters=2000, lr=1e-3,
                 lr_final=1e-5, lbfgs_iters=0, print_every=1000)
    with torch.no_grad():
        err = rel_l2(model(to_tensor(t_test)), exact(t_test))
    results["hard" if hard else "soft"] = (hist.final_loss, err)
    print("hard" if hard else "soft", "final loss %.3e   rel. L2 error %.3e" % (hist.final_loss, err))

Now compute the smallest value of the *soft* loss over affine functions $y=A+Bt$ on the same collocation
points (use `scipy.optimize.minimize`) and compare it with the final soft loss. Then add 300 L-BFGS
iterations (`lbfgs_iters=300`) to the soft run. What happens, and why?

In [ ]:
from scipy.optimize import minimize
tc = uniform(100, 0.0, T)

def soft_affine(p):
    A, B = p
    # TODO: return the discrete soft loss of y = A + B t
    return ...

best = minimize(soft_affine, [0.05, 0.0], method="Nelder-Mead",
                options=dict(xatol=1e-12, fatol=1e-16, maxiter=10000))
print("best affine soft loss %.4e at A=%.4f, B=%.4f" % (best.fun, *best.x))

In [ ]:
# check your answer
assert abs(best.fun - 6.92e-3) < 1e-4, "the affine minimum should be about 6.92e-3"
soft_loss, soft_err = results["soft"]
print("soft final loss / affine minimum = %.3f" % (soft_loss / best.fun))
print("Task 2: affine minimum correct; is the soft sigmoid run on the plateau (within 10%)?",
      abs(soft_loss / best.fun - 1) < 0.10)

## Task 3 - van der Pol with hard initial conditions

$y'' - \mu(1-y^2)y' + y = 0$, $\mu=0.5$, $y(0)=1$, $y'(0)=0$, $t\in[0,10]$.
Use the trial solution $y_\theta = 1 + \varphi(t)N_\theta(t)$ with the bounded multiplier
$\varphi(t)=t^2/(1+t^2)$, a network with two hidden layers of 32 $\tanh$ neurons, 200 collocation points,
3000 Adam iterations and at most 300 L-BFGS iterations. Compare with the DOP853 reference.

In [ ]:
MU, TV = 0.5, 10.0
t_v = np.linspace(0.0, TV, 1001)
ref = ivp_reference(lambda t, z: [z[1], MU * (1 - z[0] ** 2) * z[1] - z[0]], (0.0, TV), [1.0, 0.0], t_v)

torch.manual_seed(1234)
net = MLP(1, 1, width=32, depth=2, activation="tanh", domain=(0.0, TV))
model = ...          # TODO: ConstrainedModel with the transform 1 + t**2 / (1 + t**2) * N(t)
t_c = to_tensor(uniform(200, 0.0, TV), requires_grad=True)

def loss_fn():
    y = model(t_c)
    dy = derivative(y, t_c, 1)
    d2y = ...        # TODO: second derivative
    r = ...          # TODO: van der Pol residual
    L = (r ** 2).mean()
    return L, {"residual": L}

hist = train(model.parameters(), loss_fn, adam_iters=3000, lr=1e-3, lr_final=1e-5, lbfgs_iters=300,
             print_every=1000)
with torch.no_grad():
    y_pred = model(to_tensor(t_v)).numpy().ravel()
err_vdp = rel_l2(y_pred, ref[:, 0])
print("relative L2 error: %.3e" % err_vdp)

In [ ]:
# check your answer
assert err_vdp < 1e-2, "with this budget the relative error should be below 1e-2"
print("Task 3: correct")

## Reflection

1. In Task 2, why does the hard formulation not show the plateau of the soft formulation?
2. Repeat Task 3 with the multiplier $t^2$ instead of $t^2/(1+t^2)$. Compare the loss at iteration 0 and the
   final error, and explain the difference (Section *Higher-order equations: the van der Pol oscillator*).
3. Time the classical reference solver (`%timeit ivp_reference(...)`) and compare with the training time of
   the PINN. Under which circumstances would you nevertheless use a PINN for an IVP?